In [15]:
import scanpy as sc
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from geneformer import EmbExtractor
from geneformer import TranscriptomeTokenizer
import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps


def gene2ensembl(gene_list,mapping_path):
    mapping_df = pd.read_csv(mapping_path,header=0)
    mapping_df_unique = mapping_df.drop_duplicates(subset=['Gene name'], keep='first')
    mapping_dict = dict(zip(mapping_df_unique['Gene name'], mapping_df_unique['ensembl ID']))
    mapped_gene_list = []
    unmapped_genes = []
    for gene in gene_list:
        if gene.startswith('ENS'):
            mapped_gene_list.append(gene)
            continue
        if gene in mapping_dict:
            mapped_gene_list.append(mapping_dict[gene])
        else:
            mapped_gene_list.append(gene)
            unmapped_genes.append(gene)
    if unmapped_genes:
        print(f"Warning: {len(unmapped_genes)} genes not found in mapping file: {unmapped_genes[:10]}{'...' if len(unmapped_genes) > 10 else ''}")
    return mapped_gene_list




def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/spatial_cluster_no_annotations",save_file_name:str="record.csv"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time/60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            print(f"资源使用记录已保存到: {full_path}")
            return result
        return wrapper
    return decorator




In [28]:
shard_inx = "2"
simple_path = f'/home/cavin/jt/benchmark/data/crc/VisiumHD_P2_shard_{shard_inx}.h5ad'
adata = sc.read_h5ad(simple_path)
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [29]:
adata.var_names

Index(['ENSG00000187608', 'ENSG00000186891', 'ENSG00000186827',
       'ENSG00000162576', 'ENSG00000205116', 'ENSG00000179403',
       'ENSG00000142609', 'ENSG00000187730', 'ENSG00000116254',
       'ENSG00000049249',
       ...
       'ENSG00000102245', 'ENSG00000165370', 'ENSG00000101981',
       'ENSG00000165509', 'ENSG00000155495', 'ENSG00000268089',
       'ENSG00000182492', 'ENSG00000130821', 'ENSG00000198910',
       'ENSG00000286265'],
      dtype='object', length=2000)

In [30]:
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [31]:
if 'n_counts' not in adata.obs.columns:
    if 'UMI count' in adata.obs.columns:
        adata.obs['n_counts'] = adata.obs['UMI count']
    else:
        # 简单的 sum 作为 n_counts
        adata.obs['n_counts'] = adata.X.sum(axis=1)

In [32]:
save_dir = f"/home/cavin/jt/benchmark/experiments/middle_data/geneformer/CRC_shard_{shard_inx}"

In [7]:
preprocess_path = os.path.join(save_dir, f"CRC_shard_{shard_inx}_gene_tokenized.h5ad")
adata.write_h5ad(preprocess_path,compression="gzip")

In [8]:
tk = TranscriptomeTokenizer(custom_attr_name_dict=None, 
                            nproc=16, 
                            model_version="V2")
tk.tokenize_data(save_dir, 
                save_dir, 
                f"CRC_shard_{shard_inx}_gene_tokenized",
                file_format="h5ad")

Tokenizing /home/cavin/jt/benchmark/experiments/middle_data/geneformer/CRC_shard_2/CRC_shard_2_gene_tokenized.h5ad


/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/home/cavin/jt/benchmark/experiments/middle_data/geneformer/CRC_shard_2/CRC_shard_2_gene_tokenized.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


In [33]:
# @measure_resources(cuda_id=0, task_des=f"Geneformer run visiumHD human CRC P2 shard {shard_inx}", save_file_name=f"shard_{shard_inx}_record.csv")
def get_emb():
    # initiate EmbExtractor
    # OF NOTE: model_version should match version of model to be used (V1 or V2) to use the correct token dictionary
    embex = EmbExtractor(model_type="CellClassifier",
                        num_classes=2,
                        emb_mode='cls',
                        max_ncells=None,
                        emb_layer=0,
                        forward_batch_size=16,
                        model_version="V2",  # OF NOTE: SET TO V1 MODEL, PROVIDE V1 MODEL PATH IN SUBSEQUENT CODE
                        nproc=4)

    # extracts embedding from input data
    # input data is tokenized rank value encodings generated by Geneformer tokenizer (see tokenizing_scRNAseq_data.ipynb)
    # example dataset for V1 model series: https://huggingface.co/datasets/ctheodoris/Genecorpus-30M/tree/main/example_input_files/cell_classification/disease_classification/human_dcm_hcm_nf.dataset
    embs = embex.extract_embs("/home/cavin/jt/benchmark/models/Geneformer-V2-104M", # example V2 fine-tuned model
                            f"{save_dir}/CRC_shard_{shard_inx}_gene_tokenized.dataset",
                            "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations",
                            f"CRC_shard_{shard_inx}_gene_tokenized_embs")
    return embs

In [34]:
embs = get_emb()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /home/cavin/jt/benchmark/models/Geneformer-V2-104M and are newly initialized: ['classifier.bias', 'classifier.weight', 'bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/10427 [00:00<?, ?it/s]

In [35]:
embs.shape

(166824, 768)

In [36]:
emb_df = pd.DataFrame(
    embs.values, 
    index=adata.obs_names,
    columns=[f"geneformer_dim_{i}" for i in range(embs.shape[1])]
)
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_no_annotations/geneformer_human_CRC_shard_{shard_inx}_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")

In [37]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial'

In [38]:
adata.obsm["X_geneformer"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 166824 × 2000
    obs: 'in_tissue', 'array_row', 'array_col', 'batch', 'shard_id'
    var: 'ensembl_id', 'gene_name'
    uns: 'spatial'
    obsm: 'spatial', 'X_geneformer'

In [39]:
adata.obsm["X_geneformer"]

array([[-0.5674544 ,  0.27601728,  0.05573234, ..., -1.472113  ,
        -0.63039833,  0.7888733 ],
       [-0.5013155 , -0.01098598,  0.1614803 , ..., -1.895644  ,
        -0.97325635,  0.27703053],
       [-0.68241113,  0.33590105,  0.00980276, ..., -1.6181703 ,
        -0.47682175,  0.6607868 ],
       ...,
       [-2.1111126 , -1.3929032 , -0.8143308 , ..., -1.9905455 ,
        -0.45388693,  0.00834924],
       [-0.9025792 , -0.25362995, -1.1411415 , ..., -2.1805522 ,
        -0.408018  ,  0.30070665],
       [-1.3659061 , -0.8063587 , -0.76312125, ..., -2.3567743 ,
        -0.39399034,  0.3678328 ]], dtype=float32)